## Imports

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model
import cmocean
import cartopy.crs as ccrs
import tqdm
import copy
from sklearn.preprocessing import PowerTransformer
import sklearn.linear_model

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_windowed(data, window_size=480, stride=60):
    """Get windowed version of data (used for computing parameter values over time)"""
    
    ## get windowed data
    data_rolling = data.rolling({"time": window_size}, center=True)
    data_windowed = data_rolling.construct(window_dim="sample", stride=stride)

    ## Remove NaN values
    if type(data) is xr.Dataset:
        da = data_windowed.to_dataarray().isel(variable=0)
    else:
        da = data_windowed
    valid_idx = ~np.isnan(da).any(["sample","member"])
    data_windowed = data_windowed.isel(time=valid_idx)

    ## drop variable name
    if type(data) is xr.Dataset:
        data_windowed = data_windowed.drop_vars("variable")
    
    ## rename window coord from time to year
    year_coord = dict(year=data_windowed.time.dt.year.values)
    data_windowed = data_windowed.rename({"time": "year"}).assign_coords(year_coord)
    
    ## rename sample coord to time (use arbitrary time, used for seasonality)
    time_coord = dict(
        time=xr.date_range(start="1850-01-01", freq="MS", periods=window_size)
    )                                                                          
    data_windowed = data_windowed.rename({"sample": "time"}).assign_coords(time_coord)
    
    return data_windowed

## Load data

In [ ]:
USE_WIDE = True

## load spatial data
forced, anom_ = src.utils.load_consolidated()

if USE_WIDE:

    ## load wide data
    forced_05, anom_05 = src.utils.load_consolidated_05()

    for v in list(forced_05):
        forced[v] = forced_05[v]
        anom_[v] = anom_05[v]

## get subset of data for total
VARNAMES = ["T", "sst", "taux", "tauy", "w"]
total = anom_[VARNAMES] + forced[VARNAMES]
# total = xr.merge([forced[[f"{v}_comp" for v in VARNAMES]], total])
# total = src.utils.get_windowed(total, stride=120)

## Load T,h (total)
Th_total = xr.open_dataset(DATA_FP / "cesm" / "Th.nc")
Th = Th_total - Th_total.mean("member")
# Th = src.utils.get_windowed(Th, stride=120)

## compute dTdx

In [ ]:
## compute diff. def of Tw
sel_lat = lambda x: x.sel(latitude=slice(-5, 5)).mean(["latitude", "longitude"])
get_Tw = lambda x: sel_lat(x.sel(longitude=slice(140, 190)))
get_Te = lambda x: sel_lat(x.sel(longitude=slice(240, 280)))
get_dTdx = lambda x: get_Tw(x) - get_Te(x)

## compute dTdx (annual mean)
dTdx = src.utils.reconstruct_wrapper(xr.merge([total["sst"], forced["sst_comp"]]), get_dTdx)
dTdx = dTdx.rename({"sst":"dTdx"})
data = xr.merge([Th["T_34"], dTdx])

## window
data = get_windowed(data, window_size=480, stride=12)

## Compare methods

### Mean/median

In [ ]:
## now, compute median/mean
dTdx_mean = data["dTdx"].mean("time")
dTdx_median = data["dTdx"].median("time")

### Power transform

In [ ]:
def fit_pt(z, get_lambda=False):
    """fit power transform to data"""
    if np.isnan(z).all():
        return np.nan

    else:
        ## fit transform
        pt = PowerTransformer(standardize=False).fit(z[:, None])

        if get_lambda:
            return pt, pt.lambdas_.item()

        else:
            return pt


def fit_transform_pt(X):
    """fit power transform to data and return Xarray"""

    ## empty xarray to hold data
    dims = copy.deepcopy(X.dims)
    X_stack = X.stack(s=dims)
    Y_stack = copy.deepcopy(X_stack)

    ## do transform
    pt, lam = fit_pt(X_stack.values, get_lambda=True)
    Y_stack.values = (
        pt.transform(X_stack.values[:, None]).flatten() * X_stack.values.std()
    )

    return Y_stack.unstack("s")

In [ ]:
Y = []
for year in tqdm.tqdm(data.year):
    Y.append(fit_transform_pt(data["dTdx"].sel(year=year)))
dTdx_pt = xr.concat(Y, dim=data.year).mean("time")

### ENSO regression

In [ ]:
data["dTdx_mean"] = data["dTdx"].mean("time")
data["sigma_T"] = data["T_34"].std("time")

In [ ]:
## Linear regression object
LR = sklearn.linear_model.LinearRegression

## empty array to hold results
coefs = []

## loop thru years
for year in data.year:

    ## get data
    X = data["sigma_T"].sel(year=year)
    Y = data[["dTdx_mean"]].sel(year=year).to_dataarray()
    Y = Y.transpose("member", ...)

    ## instantiate model
    mod = LR(fit_intercept=True)
    mod.fit(X=X.values[:, None], y=Y.values)

    ## empty array to hold coefficients
    coefs_y = xr.ones_like(Y.isel(member=slice(None, 2))).rename({"member": "degree"})

    ## get coefficients
    coefs_y.values = np.concatenate([mod.intercept_[:, None], mod.coef_], axis=1).T

    ## track
    coefs.append(coefs_y)

## put coefs in array
coefs = xr.concat(coefs, dim=data.year).squeeze(drop=True)
coefs0 = coefs.sel(degree=0)

### Plot

In [ ]:
norm = lambda x : x-x.isel(year=0).median("member")

fig,ax = plt.subplots(figsize=(4,3.5))

for i, dTdx_ in enumerate([dTdx_mean, dTdx_median, dTdx_pt]):
    for q, lw in zip([.5,.1,.9], [2,.5,.5]):

        ax.plot(dTdx_.year, norm(dTdx_).quantile(q=q, dim="member"), lw=lw, c=sns.color_palette()[i])

ax.plot(coefs0.year, coefs0-coefs0.isel(year=0), lw=2, c=sns.color_palette()[i+1])

ax.axvline(2010, c="k", ls="--", lw=.8)

plt.show()